In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("LearningPlatform").getOrCreate()


In [2]:
raw_users = [
("U001","Amit","28","Hyderabad","AI,ML,Cloud"),
("U002","Neha","Thirty","Delhi","Testing"),
("U003","Ravi",None,"Bangalore",["Data","Spark"]),
("U004","Pooja","29","Mumbai","AI|ML"),
("U005","", "31","Chennai",None)
]
user_schema = StructType([
    StructField("user_id", StringType()),
    StructField("name", StringType()),
    StructField("age_raw", StringType()),
    StructField("city", StringType()),
    StructField("skills_raw", StringType())
])

In [3]:
users_df = spark.createDataFrame(raw_users, user_schema) \
.withColumn("age",
    when(col("age_raw").rlike("^[0-9]+$"), col("age_raw").cast("int"))
) \
.withColumn("skills",
    split(regexp_replace(col("skills_raw"), "[|]", ","), ",")
) \
.withColumn("name",
    when(col("name") == "", None).otherwise(col("name"))
) \
.drop("age_raw", "skills_raw")


In [4]:
raw_courses = [
("C001","PySpark Mastery","Data Engineering","Advanced","₹9999"),
("C002","AI for Testers","QA","Beginner","8999"),
("C003","ML Foundations","AI","Intermediate",None),
("C004","Data Engineering Bootcamp","Data","Advanced","₹14999")
]

In [5]:
courses_df = spark.createDataFrame(raw_courses,
["course_id","course_name","category","level","price_raw"]
).withColumn(
    "price",
    regexp_replace(col("price_raw"), "[^0-9]", "").cast("int")
).drop("price_raw")

In [6]:
raw_enrollments = [
("U001","C001","2024-01-05"),
("U002","C002","05/01/2024"),
("U003","C001","2024/01/06"),
("U004","C003","invalid_date"),
("U001","C004","2024-01-10"),
("U005","C002","2024-01-12")
]

In [7]:
enrollments_df = spark.createDataFrame(raw_enrollments,
["user_id","course_id","date_raw"]
).withColumn(
    "enroll_date",
    coalesce(
        to_date("date_raw","yyyy-MM-dd"),
        to_date("date_raw","dd/MM/yyyy"),
        to_date("date_raw","yyyy/MM/dd")
    )
).drop("date_raw")

In [8]:
raw_activity = [
("U001","login,watch,logout","{'device':'mobile'}",120),
("U002",["login","watch"],"device=laptop",90),
("U003","login|logout",None,30),
("U004",None,"{'device':'tablet'}",60),
("U005","login","{'device':'mobile'}",15)
]

In [15]:
raw_activity = [
("U001","login,watch,logout","{'device':'mobile'}",120),
("U002","login,watch","device=laptop",90), # Corrected to a string
("U003","login|logout",None,30),
("U004",None,"{'device':'tablet'}",60),
("U005","login","{'device':'mobile'}",15)
]

activity_df = spark.createDataFrame(
raw_activity,
["user_id","actions_raw","metadata","time_spent"]
).withColumn(
"actions",
split(regexp_replace(col("actions_raw"), "[|]", ","), ",")
)

In [11]:
from pyspark.sql.functions import broadcast

user_enroll_df = users_df.join(enrollments_df, "user_id", "inner")

full_df = user_enroll_df.join(
    broadcast(courses_df), "course_id", "inner"
)

In [12]:

full_df = full_df.filter(col("enroll_date").isNotNull())



In [16]:
full_df.groupBy("course_name").count()

full_df.groupBy("course_name") \
.agg(sum("price").alias("total_revenue")) # Added missing parenthesis

full_df.join(activity_df,"user_id") \
.groupBy("course_name") \
.agg(avg("time_spent").alias("avg_time"))

full_df.groupBy("user_id").count()
# Removed incomplete line: users_df.join(activity_df, "user_i")

users_df.join(activity_df, "user_id", "left_anti")

DataFrame[user_id: string, name: string, city: string, age: int, skills: array<string>]

In [17]:
w = Window.orderBy(desc("total_time"))

activity_df.groupBy("user_id") \
.agg(sum("time_spent").alias("total_time")) \
.withColumn("rank", rank().over(w))

w = Window.partitionBy("course_id").orderBy("enroll_date") \
.rowsBetween(Window.unboundedPreceding, Window.currentRow)

full_df.withColumn("running_revenue", sum("price").over(w))

w = Window.partitionBy("course_id").orderBy(desc("time_spent"))

full_df.join(activity_df,"user_id") \
.withColumn("rnk", rank().over(w)) \
.filter(col("rnk") <= 2)


DataFrame[user_id: string, course_id: string, name: string, city: string, age: int, skills: array<string>, enroll_date: date, course_name: string, category: string, level: string, price: int, actions_raw: string, metadata: string, time_spent: bigint, actions: array<string>, rnk: int]

In [18]:
engagement_df = activity_df.withColumn(
"engagement_level",
when(col("time_spent") >= 100, "High")
.when(col("time_spent") >= 40, "Medium")
.otherwise("Low")
)


In [19]:
full_df.groupBy("course_name") \
.agg(sum("price").alias("revenue")) \
.orderBy(desc("revenue"))

DataFrame[course_name: string, revenue: bigint]

In [20]:
users_df.join(activity_df,"user_id") \
.orderBy("city", desc("time_spent"))

DataFrame[user_id: string, name: string, city: string, age: int, skills: array<string>, actions_raw: string, metadata: string, time_spent: bigint, actions: array<string>]

In [21]:
enrolled_users = enrollments_df.select("user_id").distinct()
active_users = activity_df.select("user_id").distinct()

# 24
enrolled_users.subtract(active_users)

# 25
enrolled_users.intersect(active_users)

DataFrame[user_id: string]